In [0]:


import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *


@dlt.table(
    name="silver_material_master",
    comment="Cleaned material master data",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_unit_cost", "unit_cost > 0")
@dlt.expect_or_drop("valid_material_id", "material_id IS NOT NULL")
def silver_material_master():
    return (
        dlt.read("bronze_material_master")
        
        .withColumn("unit_cost_clean", 
                    when(col("unit_cost").rlike("^[0-9]+\\.?[0-9]*$"), 
                         col("unit_cost").cast(DecimalType(10, 2)))
                    .otherwise(None))
        
        .withColumn("last_updated_date", to_date(col("last_updated"), "yyyy-MM-dd"))
        .withColumn("lead_time_days_int", col("lead_time_days").cast(IntegerType()))
        .withColumn("safety_stock_int", col("safety_stock").cast(IntegerType()))
        .withColumn("reorder_level_int", col("reorder_level").cast(IntegerType()))
        
        .withColumn("material_id_clean", trim(upper(col("material_id"))))
        .withColumn("material_name_clean", trim(col("material_name")))
        .withColumn("plant_clean", trim(col("plant")))
        .withColumn("status_clean", trim(col("status")))
        
        .withColumn("silver_processed_timestamp", current_timestamp())
        
        .select(
            col("material_id_clean").alias("material_id"),
            col("material_name_clean").alias("material_name"),
            col("category"),
            col("plant_clean").alias("plant"),
            col("status_clean").alias("status"),
            col("unit_cost_clean").alias("unit_cost"),
            col("supplier_name"),
            col("country"),
            col("last_updated_date").alias("last_updated"),
            col("lead_time_days_int").alias("lead_time_days"),
            col("safety_stock_int").alias("safety_stock"),
            col("reorder_level_int").alias("reorder_level"),
            col("ingestion_timestamp"),
            col("silver_processed_timestamp")
        )
    )


@dlt.table(
    name="silver_material_master_quarantine",
    comment="Invalid records"
)
def silver_quarantine():
    return (
        dlt.read("bronze_material_master")
        .where(~col("unit_cost").rlike("^[0-9]+\\.?[0-9]*$"))
        .withColumn("quarantine_reason", lit("Invalid unit_cost"))
        .withColumn("quarantine_timestamp", current_timestamp())
    )